<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/03_machine_learning/scikit-learn/tools-for-working-with-data/datasets.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/03_machine_learning/scikit-learn/tools-for-working-with-data/datasets.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.

# Working with Data: Synthetic Generators and File Round-Trips

## Context
Every other notebook in this repo starts from data that's already sitting in a clean DataFrame. In practice, getting there is its own small skill: sometimes you fabricate a synthetic dataset by hand to prototype an idea, sometimes you lean on a parametrized generator like `make_classification` for a quick, controllable dataset, and sometimes you're round-tripping a CSV or JSON export from a monitoring system or data warehouse. This notebook is about that plumbing, not a specific algorithm.

## Objectives
- Build a synthetic infra-metrics dataset by hand with NumPy and inspect it like a real DataFrame.
- Use `make_classification` to generate a parametrized, controllable dataset, then actually train and score a baseline model on it.
- Round-trip a dataset through CSV and JSON files, the way you'd receive a real export.

## Expected Outcome
- A reusable mental model for how data actually gets into `X`/`y` form before you reach for a specific ML algorithm.
- A synthetic "will this trigger a page?" dataset you generated yourself, with a baseline model already fit against it.

### 1. Build a Synthetic Dataset by Hand
Sometimes there's no dataset yet — you're prototyping before real telemetry exists. Here we fabricate a small table of server metrics with NumPy, the same way several other notebooks in this repo do.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 200

metrics_df = pd.DataFrame({
    "cpu_pct": np.clip(rng.normal(55, 15, n), 0, 100),
    "mem_pct": np.clip(rng.normal(60, 12, n), 0, 100),
    "latency_ms": rng.exponential(80, n),
    "error_count": rng.poisson(2, n),
})

metrics_df.head()

,cpu_pct,mem_pct,latency_ms,error_count
0,59.570756,64.050895,46.728694,4
1,39.400238,76.889782,174.023633,4
2,66.256768,61.087019,17.265807,1
3,69.108471,67.727266,310.873130,0
4,25.734472,35.397935,174.255048,6


Once it's a DataFrame, the usual pandas inspection tools apply — `.describe()` gives a quick numeric summary without writing a single plotting call.

In [2]:
metrics_df.describe()

,cpu_pct,mem_pct,latency_ms,error_count
count,200.000000,200.000000,200.000000,200.00000
mean,54.543239,60.240931,87.342708,2.02000
std,13.228994,12.234058,86.627758,1.39619
min,23.019306,29.200099,0.714756,0.00000
25%,45.151416,51.644245,25.992525,1.00000
50%,54.222187,60.725396,58.662491,2.00000
75%,63.025307,67.669621,118.771534,3.00000
max,98.707937,94.860806,493.527972,8.00000


### 2. Generate a Parametrized Dataset with `make_classification`
Hand-rolling columns one by one gets tedious once you want dozens of features with controlled informativeness, class imbalance, and noise. Scikit-learn's `make_classification` (and its regression counterpart, `make_regression`) generate a fully-specified synthetic dataset in one call — useful for prototyping a model before real labeled data exists.

In [3]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, n_features=4, n_informative=3, n_redundant=1,
    weights=[0.8, 0.2],  # most requests are fine; ~20% should page someone
    random_state=42,
)

synthetic_df = pd.DataFrame(X, columns=["cpu_pct", "mem_pct", "req_rate", "error_rate"])
synthetic_df["will_page"] = y

synthetic_df.head()

,cpu_pct,mem_pct,req_rate,error_rate,will_page
0,-1.634716,0.120040,-0.955870,-0.648328,0
1,0.445277,-1.986198,0.219744,1.869351,0
2,-1.892225,-1.032171,1.541761,-0.175732,0
3,0.081571,-2.216266,-0.718966,2.066282,0
4,-1.298140,-0.547365,-0.174167,-0.056450,0


Generating a dataset is only useful if you actually train something on it — otherwise it's dead code. Let's fit a quick baseline classifier and see how it does.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

print(classification_report(y_test, clf.predict(X_test), target_names=["no_page", "will_page"]))

              precision    recall  f1-score   support

     no_page       0.95      0.98      0.97       246
   will_page       0.91      0.76      0.83        54

    accuracy                           0.94       300
   macro avg       0.93      0.87      0.90       300
weighted avg       0.94      0.94      0.94       300



### 3. Round-Tripping Through Files
Most real data doesn't arrive as a Python object — it arrives as a CSV export from a dashboard or a JSON blob from an API. Once it's back in a DataFrame, scikit-learn doesn't care which format it came from.

In [5]:
metrics_df.to_csv("server_metrics.csv", index=False)
metrics_df.to_json("server_metrics.json", orient="records")

csv_df = pd.read_csv("server_metrics.csv")
json_df = pd.read_json("server_metrics.json")

print(f"CSV round-trip preserves values: {np.allclose(csv_df.values, metrics_df.values)}")
print(f"JSON round-trip preserves values: {np.allclose(json_df.values, metrics_df.values)}")

CSV round-trip preserves values: True
JSON round-trip preserves values: True


## Summary
Getting data into a trainable shape is a skill independent of any specific algorithm:

- **Hand-rolled synthetic data** is the fastest way to prototype when no real dataset exists yet.
- **`make_classification` / `make_regression`** give you a controllable, parametrized dataset — tune class imbalance, noise, and informative features without writing the generator yourself.
- **File round-trips** (CSV, JSON, and in production likely Parquet) are how most real data actually arrives — the loading step is often more work than the modeling step.

For real-world datasets beyond what you generate yourself, scikit-learn also ships `fetch_openml()` and dataset-specific fetchers like `fetch_california_housing()`, which download data on first use — handy once you've outgrown synthetic prototypes.